# LoRA vs LoRA+ Experimental Evaluation

## RoBERTa-base + QQP

### 1. Project Setup and GitHub Integration

Clone the LoRA-Plus project repository from GitHub into Google Colab to utilize GPU resources for model training.

Experiment results are saved in the repository's `results/` directory and can be committed and pushed to GitHub for persistent storage.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/Ramprasath-Meganathan/LoRA-Plus.git"
PROJECT_DIR = Path("/content/LoRA-Plus")

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True
    )
else:
    print("Repository already exists. Using existing files.")

for filename in [
    "config.py",
    "model.py",
    "utils.py",
    "train.py",
    "evaluate_results.py"
]:
    assert (PROJECT_DIR / filename).is_file(), (
        f"Missing project file: {filename}"
    )

os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project:", os.getcwd())

## 2. Install Required Dependencies

Install the required libraries from the repository's `requirements.txt` file for model fine-tuning, LoRA/LoRA+ optimization, dataset processing, and evaluation.

Remove incompatible `torchao` installations to ensure compatibility with PEFT in Google Colab.

In [ ]:
%pip install -r requirements.txt
%pip uninstall -y torchao

## 3. Experiment Configuration and GPU Verification

Select RoBERTa-base and the Quora Question Pairs (QQP) dataset.

The experiment uses three training epochs, nine hyperparameter screening configurations, and three random seeds (42, 43, 44) for final evaluation.

Verify that a GPU is available before starting training.

In [ ]:
import torch

MODEL_NAME = "roberta"
DATASET_NAME = "qqp"

assert torch.cuda.is_available(), (
    "GPU unavailable. Enable a GPU runtime in Colab."
)

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)
print("GPU:", torch.cuda.get_device_name(0))

## 4. Import Shared Project Modules

Import the reusable configuration, dataset preparation, training, and evaluation functions from the project's Python modules.

These modules provide a consistent experimental pipeline for standard LoRA and LoRA+.

In [ ]:
from config import (
    COMMON_CONFIG,
    DATASET_CONFIGS,
    LORA_LEARNING_RATES,
    LORAPLUS_CONFIGS,
    get_output_dir
)

from utils import prepare_dataset
from train import run_experiment

from evaluate_results import (
    select_best,
    summarize_final,
    plot_convergence
)

print("All project modules imported successfully!")
print("Results directory:", get_output_dir(MODEL_NAME, DATASET_NAME))

assert COMMON_CONFIG["epochs"] == 3
assert COMMON_CONFIG["seeds"] == [42, 43, 44]
assert DATASET_CONFIGS["qqp"]["primary_metric"] == "f1"

## 5. QQP Dataset Loading and Tokenization

Load the GLUE Quora Question Pairs dataset and tokenize both questions using the RoBERTa tokenizer.

Use a fixed 50,000-example training subset and the complete validation split. Prepare the dataset once and reuse it across all training configurations.

In [ ]:
tokenizer, train_dataset, eval_dataset, data_collator = (
    prepare_dataset(
        model_name=MODEL_NAME,
        dataset_name=DATASET_NAME,
        pilot=False
    )
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))

assert len(train_dataset) == 50000
assert len(eval_dataset) == 40430

## 6. Training Pipeline Sanity Test

Run a short 50-step LoRA+ training experiment to verify model initialization, adapter configuration, optimizer learning rates, GPU execution, and validation metrics.

This sanity test is excluded from the official screening and final experiment results.

In [ ]:
test = run_experiment(
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    lam=4,
    seed=42,
    epochs=3,
    base_lr=1e-4,
    stage="test",
    max_steps=50
)

print("Training steps:", test["total_steps"])
print("Validation accuracy:", test["final_acc"])
print("Validation binary F1:", test["final_f1"])

assert test["total_steps"] == 50

## 7. Standard LoRA Hyperparameter Screening

Evaluate standard LoRA using three learning rates: 1e-4, 2e-4, and 4e-4.

Both adapter matrices A and B use equal learning rates. Each configuration is trained for three epochs using random seed 42.

In [ ]:
for lr in LORA_LEARNING_RATES:

    print(f"\nRunning standard LoRA: LR={lr}")

    run_experiment(
        model_name=MODEL_NAME,
        dataset_name=DATASET_NAME,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        lam=1,
        seed=COMMON_CONFIG["screening_seed"],
        epochs=COMMON_CONFIG["epochs"],
        base_lr=lr,
        stage="screening"
    )

## 8. LoRA+ Hyperparameter Screening

Evaluate six LoRA+ configurations using base learning rates of 1e-4 and 2e-4 with learning-rate ratios of 4×, 8×, and 16×.

Matrix B uses a larger learning rate than matrix A, according to:

η_B = λ × η_A

All configurations use the same training budget and random seed as standard LoRA.

In [ ]:
for lr, ratio in LORAPLUS_CONFIGS:

    print(
        f"\nRunning LoRA+: "
        f"A LR={lr}, B LR={lr * ratio}, ratio={ratio}"
    )

    run_experiment(
        model_name=MODEL_NAME,
        dataset_name=DATASET_NAME,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        lam=ratio,
        seed=COMMON_CONFIG["screening_seed"],
        epochs=COMMON_CONFIG["epochs"],
        base_lr=lr,
        stage="screening"
    )

## 9. Best Hyperparameter Selection

Select the best-performing standard LoRA and LoRA+ configurations based on final validation binary F1 after three epochs.

Validation loss is used as a tie-breaker when configurations achieve identical F1 scores.

In [ ]:
best_lora, best_loraplus = select_best(
    MODEL_NAME,
    DATASET_NAME
)

BEST_LORA_LR = float(best_lora["eta_A"])
BEST_LORAPLUS_LR = float(best_loraplus["eta_A"])
BEST_LAM = int(best_loraplus["lam"])

print("\nSelected standard LoRA:")
print("A LR:", BEST_LORA_LR)
print("B LR:", BEST_LORA_LR)

print("\nSelected LoRA+:")
print("A LR:", BEST_LORAPLUS_LR)
print("B LR:", BEST_LORAPLUS_LR * BEST_LAM)
print("Ratio:", BEST_LAM)

## 10. Final Multi-Seed Experiments

Evaluate the selected standard LoRA and LoRA+ configurations across three random seeds: 42, 43, and 44.

Each run initializes a fresh pretrained model and trains for three epochs using the same dataset and experimental settings.

In [ ]:
for seed in COMMON_CONFIG["seeds"]:

    print(f"\nFinal LoRA run — seed {seed}")

    run_experiment(
        model_name=MODEL_NAME,
        dataset_name=DATASET_NAME,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        lam=1,
        seed=seed,
        epochs=COMMON_CONFIG["epochs"],
        base_lr=BEST_LORA_LR,
        stage="final"
    )

    print(f"\nFinal LoRA+ run — seed {seed}")

    run_experiment(
        model_name=MODEL_NAME,
        dataset_name=DATASET_NAME,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        lam=BEST_LAM,
        seed=seed,
        epochs=COMMON_CONFIG["epochs"],
        base_lr=BEST_LORAPLUS_LR,
        stage="final"
    )

## 11. Final Performance Summary

Aggregate the final experimental results across three random seeds.

Report mean and standard deviation for validation accuracy, binary F1, macro-F1, validation loss, training runtime, GPU memory usage, and trainable parameter count.

In [ ]:
summary = summarize_final(
    MODEL_NAME,
    DATASET_NAME
)

display(summary)

## 12. Convergence Analysis

Compare the convergence behavior of standard LoRA and LoRA+ using validation binary F1 against optimizer steps.

Plot the mean performance across three random seeds with standard-deviation bands to analyze training efficiency and convergence stability.

In [ ]:
plot_convergence(
    MODEL_NAME,
    DATASET_NAME
)

## 13. Experiment Results and GitHub Version Control

Verify that the completed QQP experiment results have been saved inside the repository's `results/roberta/qqp/` directory.

Commit and push CSV, JSON, and convergence plots to GitHub for persistent storage. Exclude large model checkpoints from version control.

In [ ]:
from pathlib import Path

root = get_output_dir(MODEL_NAME, DATASET_NAME)

print("Results directory:", root)

for path in sorted(root.rglob("*")):
    if path.is_file() and "checkpoints" not in path.parts:
        print(path.relative_to(root))